In [1]:
%%capture
# Data science
import numpy as np
import pandas as pd

# Data visualization
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
%matplotlib inline

from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, roc_auc_score, f1_score, \
    mean_squared_error, mean_absolute_error, r2_score
from lightgbm import LGBMClassifier, plot_importance, LGBMRegressor

import optuna
from optuna.samplers import TPESampler

import math
import warnings
import requests
from datetime import date
from mkl_random.mklrand import shuffle
warnings.filterwarnings('ignore')

# Initialization
## Import

In [2]:
df= pd.read_csv("environment_training.csv")

In [3]:
print(f'The Train dataset has {df.shape[0]} rows and {df.shape[1]} columns')
df.describe()

The Train dataset has 63524 rows and 36 columns


,total_parking_minutes,metar_temperature_c,metar_relative_humidity,metar_dew_point_c,metar_wind_speed_kn,metar_visibility_mi,metar_hour_precipitation,sea_salt_aerosol_003_05_mixing_ratio,sea_salt_aerosol_05_5_mixing_ratio,sea_salt_aerosol_5_20_mixing_ratio,...,h2o2,formaldehyde,hno3,nitrogen_monoxide_mass_mixing_ratio,nitrogen_dioxide_mass_mixing_ratio,oh,organic_nitrates,specific_humidity,sulphur_dioxide_mass_mixing_ratio,temperature
count,63524.000000,63488.000000,63487.000000,63487.000000,63524.000000,63524.000000,63516.000000,6.352400e+04,6.352400e+04,6.352400e+04,...,6.352400e+04,6.352400e+04,6.352400e+04,6.352400e+04,6.352400e+04,6.352400e+04,6.352400e+04,63524.000000,6.352400e+04,63524.000000
mean,30445.508355,16.513813,67.245746,11.180438,7.139854,6.356767,0.000813,9.206393e-11,7.664622e-09,3.308561e-09,...,6.629142e-10,2.376040e-09,1.626462e-09,3.925087e-09,1.247501e-08,3.724992e-14,1.858148e-09,0.009947,1.330595e-08,290.373316
std,8381.243070,8.272421,18.257788,7.296641,1.786242,1.617738,0.003563,7.181926e-11,6.075969e-09,3.633858e-09,...,5.418851e-10,2.134027e-09,2.298840e-09,9.240778e-09,8.910395e-09,2.274816e-14,1.653642e-09,0.004227,3.646022e-08,7.371558
min,1.050000,-16.453889,1.381661,-20.239208,0.000000,0.190000,0.000000,4.985393e-15,3.769741e-13,2.662750e-13,...,4.270074e-12,5.129457e-11,4.216254e-12,1.529446e-12,8.855178e-11,2.215035e-16,9.587580e-12,0.000784,4.140415e-11,256.088154
25%,25164.791667,9.563114,63.453113,5.305009,5.961638,5.704962,0.000000,4.456727e-11,3.654565e-09,9.779735e-10,...,2.212862e-10,8.996958e-10,6.067321e-10,6.405569e-10,7.377388e-09,2.114360e-14,9.880509e-10,0.006521,1.950105e-09,284.933605
50%,28959.891667,16.858056,72.869789,10.891892,7.009490,5.985442,0.000000,7.450668e-11,6.149368e-09,2.163902e-09,...,5.547921e-10,1.751915e-09,9.937248e-10,1.437501e-09,1.045865e-08,3.549122e-14,1.457708e-09,0.009288,4.157565e-09,290.953202
75%,35905.120833,23.251531,79.206831,16.176368,8.160092,6.157879,0.000000,1.179378e-10,9.823090e-09,4.284726e-09,...,9.950975e-10,3.374727e-09,1.525822e-09,3.175689e-09,1.452690e-08,4.857813e-14,2.214340e-09,0.012817,1.047581e-08,296.443402
max,44640.000000,39.650294,100.000000,27.066209,23.125000,35.187595,0.104874,1.772156e-09,1.512630e-07,1.013043e-07,...,9.425839e-09,5.642228e-08,1.097212e-07,1.928283e-07,1.820872e-07,2.838620e-13,4.895763e-08,0.026182,8.196594e-07,311.932815


In [4]:
display('Datas:', df)

'Datas:'

,aircraft_id,year_month,month_start_date,total_parking_minutes,metar_temperature_c,metar_relative_humidity,metar_dew_point_c,metar_wind_speed_kn,metar_visibility_mi,metar_hour_precipitation,...,h2o2,formaldehyde,hno3,nitrogen_monoxide_mass_mixing_ratio,nitrogen_dioxide_mass_mixing_ratio,oh,organic_nitrates,specific_humidity,sulphur_dioxide_mass_mixing_ratio,temperature
0,a414b2,2017-04,2017-04-01T00:00:00Z,41220.150000,30.718293,30.899519,9.558831,5.710531,2.196935,0.0,...,1.391024e-09,5.049084e-09,5.711211e-09,2.694732e-08,3.239839e-08,9.621187e-14,1.688966e-09,0.007535,5.227935e-08,302.226554
1,a414b2,2021-07,2021-07-01T00:00:00Z,28587.733333,29.397745,78.179996,24.765060,6.290009,2.414297,0.0,...,1.169712e-09,2.717347e-09,2.222564e-09,6.977001e-09,1.693530e-08,6.740166e-14,8.613307e-10,0.018132,1.506060e-08,301.692884
2,41ef34,2024-01,2024-01-01T00:00:00Z,29868.166667,15.522649,76.737610,10.499426,3.765631,1.029053,0.0,...,7.315543e-10,5.885936e-09,4.869761e-09,2.471157e-08,4.266760e-08,2.792927e-14,7.472720e-09,0.008187,5.721327e-08,287.567593
3,e2d345,2019-05,2019-05-01T00:00:00Z,18139.383333,30.951416,42.349314,15.166188,7.073745,5.594613,0.0,...,2.941217e-09,2.050597e-08,1.112867e-08,8.965584e-10,4.222114e-08,1.126736e-13,2.468868e-08,0.014630,1.843788e-08,301.816769
4,cfa8ab,2025-07,2025-07-01T00:00:00Z,26916.333333,27.367780,81.524175,23.630275,7.467336,2.774049,0.0,...,8.699552e-10,3.026269e-09,2.116588e-09,4.818643e-09,1.728175e-08,7.366147e-14,1.265471e-09,0.018309,1.331511e-08,299.727528
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63519,661e37,2018-11,2018-11-01T00:00:00Z,43200.000000,12.047059,82.966676,9.064706,6.873529,5.343588,0.0,...,8.975586e-11,8.468723e-10,8.115343e-10,2.796081e-09,1.363158e-08,1.128950e-14,1.126368e-09,0.007033,1.470342e-09,285.244050
63520,4b3253,2017-09,2017-09-01T00:00:00Z,43200.000000,27.310231,79.124785,22.943894,3.006579,5.950987,0.0,...,1.508886e-09,5.082439e-09,5.772846e-11,3.342279e-11,7.785108e-10,1.736884e-15,1.074707e-09,0.017245,5.628162e-10,301.650542
63521,b8edf3,2018-07,2018-07-01T00:00:00Z,44640.000000,26.648810,79.650060,22.431548,3.413690,6.141250,0.0,...,1.007546e-09,3.644419e-09,3.031872e-11,2.182628e-11,5.608379e-10,1.656447e-15,7.825175e-10,0.017126,5.198026e-10,300.423651
63522,2d6473,2025-09,2025-09-01T00:00:00Z,80.266667,26.000000,78.670000,22.000000,2.500000,6.210000,0.0,...,1.614644e-09,1.329852e-08,1.985568e-10,7.445210e-11,7.755650e-09,7.959141e-16,4.542368e-09,0.017704,2.995566e-09,299.071251


## Summary

In [5]:
def summary(df):
    summ = pd.DataFrame(df.dtypes, columns=['data type'])
    summ['#missing'] = df.isnull().sum().values
    summ['Duplicate'] = df.duplicated().sum()
    summ['#unique'] = df.nunique().values
    desc = pd.DataFrame(df.describe(include='all').transpose())
    if 'min' in desc.columns:
        summ['min'] = desc['min'].values
    if 'max' in desc.columns:
        summ['max'] = desc['max'].values
    if 'mean' in desc.columns:
        summ['avg'] = desc['mean'].values
    if 'std' in desc.columns:
        summ['std dev'] = desc['std'].values
    if 'top' in desc.columns:
        summ['top value'] = desc['top'].values
    if 'freq' in desc.columns:
        summ['Freq'] = desc['freq'].values

    return summ

In [6]:
summary(df).style.background_gradient()

,data type,#missing,Duplicate,#unique,min,max,avg,std dev,top value,Freq
aircraft_id,str,0,0,758,nan,nan,nan,nan,cead5b,129
year_month,str,0,0,146,nan,nan,nan,nan,2025-07,735
month_start_date,str,0,0,146,nan,nan,nan,nan,2025-07-01T00:00:00Z,735
total_parking_minutes,float64,0,0,56734,1.050000,44640.000000,30445.508355,8381.243070,nan,nan
metar_temperature_c,float64,36,0,59333,-16.453889,39.650294,16.513813,8.272421,nan,nan
metar_relative_humidity,float64,37,0,59283,1.381661,100.000000,67.245746,18.257788,nan,nan
metar_dew_point_c,float64,37,0,59280,-20.239208,27.066209,11.180438,7.296641,nan,nan
metar_wind_speed_kn,float64,0,0,59272,0.000000,23.125000,7.139854,1.786242,nan,nan
metar_visibility_mi,float64,0,0,59175,0.190000,35.187595,6.356767,1.617738,nan,nan
metar_hour_precipitation,float64,8,0,9744,0.000000,0.104874,0.000813,0.003563,nan,nan


# Diagrams
## Showplot

In [7]:
def showplot(columnname, df):
    plt.rcParams['figure.facecolor'] = 'white'
    plt.rcParams['axes.facecolor'] = 'white'
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax = ax.flatten()

    value_counts = df[columnname].value_counts()
    labels = value_counts.index.tolist()

    colors = [to_hex(c) for c in sns.color_palette("Set2", len(labels))]
    color_map = dict(zip(labels, colors))

    # Donut
    ax[0].pie(
        value_counts,
        autopct='%1.1f%%',
        textprops={'size': 9, 'color': 'white', 'fontweight': 'bold'},
        colors=[color_map[label] for label in labels],
        wedgeprops=dict(width=0.35),
        startangle=80,
        pctdistance=0.85
    )
    centre_circle = plt.Circle((0, 0), 0.6, fc='white')
    ax[0].add_artist(centre_circle)

    # Bar chart
    sns.countplot(
        data=df,
        y=columnname,
        hue=columnname,
        order=labels,
        hue_order=labels,
        palette=color_map,
        saturation=1,
        dodge=False,
        ax=ax[1],
        legend=False
    )

    for i, v in enumerate(value_counts):
        ax[1].text(v + 1, i, str(v), color='black', fontsize=10, va='center')

    sns.despine(left=True, bottom=True)
    ax[1].set_ylabel(None)
    ax[1].set_xlabel("")
    ax[1].set_xticks([])
    fig.suptitle(columnname, fontsize=15, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 0.85, 1])
    plt.show()

## Dist

In [8]:
def dist(datasets, labels, drop_columns=None, rows=-1, cols=4):
    colors = ['#05b0a3', '#d68c78', '#c7f2a6', '#21deb2', '#deb221', '#de87cb']
    columns_list = datasets[0].select_dtypes(include=['float64', 'int64']).drop(columns=drop_columns).columns
    rows = math.ceil(len(columns_list) / cols)
    fig, axs = plt.subplots(rows, cols, figsize=(24, 5*rows))
    subtitle = f'Distribution for numerical features: {labels[0]}'
    for i in range(1, len(datasets)):
        subtitle += f' vs {labels[i]}'
    plt.suptitle(subtitle, fontsize=16, fontweight='bold')
    axs = axs.flatten()

    for i, col in enumerate(columns_list):
        for j in range(len(datasets)):
            sns.kdeplot(datasets[j][col], ax=axs[i], fill=True, alpha=0.5, linewidth=0.5, color=colors[j], label=labels[j])
            axis_title = f'{col}: {labels[0]} skewness: {datasets[0][col].skew():.2f}'
            for j in range(1, len(datasets)):
                axis_title += f'\n{labels[j]} skewness: {datasets[j][col].skew():.2f}'
            axs[i].set_title(axis_title)
        axs[i].legend()

        plt.tight_layout()

## Feature distribution

In [9]:
def feature_distribution(feature, val_feature, df):
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    ax1 = axes[0]
    df_sort = df.groupby(val_feature)[feature].mean().sort_values(ascending=False).index
    sns.barplot(x=feature, y=val_feature, data=df, palette='light:#4caba4_r', order=df_sort,
                estimator=np.mean, ci=None, errwidth=0, ax=ax1)
    for p in ax1.patches:
        ax1.annotate(f'{p.get_width():.2f}', (p.get_x() + p.get_width() / 2., p.get_y() + p.get_height()),
            ha='center', va='center', xytext=(0, 20), textcoords='offset points', fontsize=10, color='black')
    ax1.set_title(f'Mean {feature} by {val_feature}')
    ax1.set_xlabel(feature)
    ax1.set_ylabel('')
    sns.despine(left=True, bottom=True, ax=ax1)

    # Violin Plot
    ax2 = axes[1]
    sns.violinplot(x=feature, y=val_feature, data=df, palette='light:#4caba4_r', order=df_sort, ax=ax2)
    ax2.set_title(f'Distribution of {feature} by {val_feature}')
    ax2.set_ylabel("")
    plt.yticks([])
    sns.despine(left=True, bottom=True, ax=ax2)
    plt.tight_layout()
    plt.show()

## Check outliers

In [10]:
def check_outliers(df, ncols=3, figsize_per_plot=(8, 3)):
    num_cols = df.select_dtypes(include=['float64', 'int64']).columns
    n = len(num_cols)

    if n == 0:
        print("Aucune colonne numérique.")
        return

    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(figsize_per_plot[0] * ncols, figsize_per_plot[1] * nrows)
    )

    if nrows == 1 and ncols == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    fig.suptitle('Outliers in the data', fontsize=18, fontweight='bold')

    for i, col in enumerate(num_cols):
        sns.boxplot(data=df, x=col, ax=axes[i])
        axes[i].set_title(col)
        axes[i].set_xlabel(col)
        axes[i].grid(False)

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    fig.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

## Correlation

In [11]:
def correlation_heatmap(datasets, labels):
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))

    for i in range(len(datasets)):
        corr = datasets[i].corr(method='pearson')
        mask = np.triu(np.ones_like(corr))
        sns.heatmap(corr, annot=True, fmt='.2f', mask=mask, cmap='copper_r', cbar=None, linewidth=2, ax=axes[i])
        axes[i].set_title(f'{labels[i]} Dataset', fontsize=16, fontweight='bold')

    plt.tight_layout()
    plt.show()

## Cross tab

In [12]:
def cross_tab(row_data, col_data, df):
    ct = pd.crosstab(df[row_data], df[col_data])
    plt.figure(figsize=(10, 5))
    sns.heatmap(ct, annot=True, cmap='Blues', fmt='d', cbar=False)
    plt.title(f'{row_data} and {col_data}')
    plt.xlabel('')
    plt.ylabel('')
    plt.show()

# EDA

In [13]:
eda_df = df.copy()
eda_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 63524 entries, 0 to 63523
Data columns (total 36 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   aircraft_id                                      63524 non-null  str    
 1   year_month                                       63524 non-null  str    
 2   month_start_date                                 63524 non-null  str    
 3   total_parking_minutes                            63524 non-null  float64
 4   metar_temperature_c                              63488 non-null  float64
 5   metar_relative_humidity                          63487 non-null  float64
 6   metar_dew_point_c                                63487 non-null  float64
 7   metar_wind_speed_kn                              63524 non-null  float64
 8   metar_visibility_mi                              63524 non-null  float64
 9   metar_hour_precipitation               

## Distribution

In [14]:
# dist([eda_df], ['EDA'], [], cols=3)

In [15]:
# check_outliers(df)

## Correlation

In [16]:
num_col_train = eda_df.select_dtypes(include=np.number)
# correlation_heatmap([num_col_train], ['EDA'])

# Data reduction
## View

In [17]:
summary(df.dropna()).style.background_gradient()

,data type,#missing,Duplicate,#unique,min,max,avg,std dev,top value,Freq
aircraft_id,str,0,0,758,nan,nan,nan,nan,870048,128
year_month,str,0,0,146,nan,nan,nan,nan,2025-07,735
month_start_date,str,0,0,146,nan,nan,nan,nan,2025-07-01T00:00:00Z,735
total_parking_minutes,float64,0,0,56724,1.050000,44640.000000,30444.295038,8371.362544,nan,nan
metar_temperature_c,float64,0,0,59332,-16.453889,39.650294,16.513721,8.272454,nan,nan
metar_relative_humidity,float64,0,0,59283,1.381661,100.000000,67.245746,18.257788,nan,nan
metar_dew_point_c,float64,0,0,59280,-20.239208,27.066209,11.180438,7.296641,nan,nan
metar_wind_speed_kn,float64,0,0,59265,0.000000,23.125000,7.140583,1.786192,nan,nan
metar_visibility_mi,float64,0,0,59169,0.190000,35.187595,6.352590,1.600460,nan,nan
metar_hour_precipitation,float64,0,0,9743,0.000000,0.104874,0.000813,0.003563,nan,nan


In [18]:
df[['month_start_date', 'year_month']]

,month_start_date,year_month
0,2017-04-01T00:00:00Z,2017-04
1,2021-07-01T00:00:00Z,2021-07
2,2024-01-01T00:00:00Z,2024-01
3,2019-05-01T00:00:00Z,2019-05
4,2025-07-01T00:00:00Z,2025-07
...,...,...
63519,2018-11-01T00:00:00Z,2018-11
63520,2017-09-01T00:00:00Z,2017-09
63521,2018-07-01T00:00:00Z,2018-07
63522,2025-09-01T00:00:00Z,2025-09


In [22]:
train_df = df.copy()
train_df['year'] = train_df['year_month'].str.slice(start=0, stop=4).astype(np.int32)
train_df['month'] = train_df['year_month'].str.slice(start=5).astype(np.int32)
train_df = train_df.drop(columns=['year_month', 'month_start_date', 'aircraft_id'])
train_df = train_df.dropna()
train_df[['year', 'month']]

,year,month
0,2017,4
1,2021,7
2,2024,1
3,2019,5
4,2025,7
...,...,...
63519,2018,11
63520,2017,9
63521,2018,7
63522,2025,9


## New computed dataframe

In [23]:
summary(train_df).style.background_gradient()

,data type,#missing,Duplicate,#unique,min,max,avg,std dev
total_parking_minutes,float64,0,32,56724,1.050000,44640.000000,30444.295038,8371.362544
metar_temperature_c,float64,0,32,59332,-16.453889,39.650294,16.513721,8.272454
metar_relative_humidity,float64,0,32,59283,1.381661,100.000000,67.245746,18.257788
metar_dew_point_c,float64,0,32,59280,-20.239208,27.066209,11.180438,7.296641
metar_wind_speed_kn,float64,0,32,59265,0.000000,23.125000,7.140583,1.786192
metar_visibility_mi,float64,0,32,59169,0.190000,35.187595,6.352590,1.600460
metar_hour_precipitation,float64,0,32,9743,0.000000,0.104874,0.000813,0.003563
sea_salt_aerosol_003_05_mixing_ratio,float64,0,32,59387,0.000000,0.000000,0.000000,0.000000
sea_salt_aerosol_05_5_mixing_ratio,float64,0,32,59396,0.000000,0.000000,0.000000,0.000000
sea_salt_aerosol_5_20_mixing_ratio,float64,0,32,59410,0.000000,0.000000,0.000000,0.000000


In [ ]:
"""final_df = sample_df.drop(columns=[
    'sample_id',
    'source_dataset',
    'location_country',
    'location_continent',
    'timestamp',
    'sample_year',
    'toxin_value_raw',
    'toxin_unit',
    'fungal_species',
    'association_type',
    'original_metadata',
    'year',
    'month',
    'day',
    'is_feed'
])"""

In [26]:
final_df = train_df.copy()
final_df.columns

Index(['total_parking_minutes', 'metar_temperature_c',
       'metar_relative_humidity', 'metar_dew_point_c', 'metar_wind_speed_kn',
       'metar_visibility_mi', 'metar_hour_precipitation',
       'sea_salt_aerosol_003_05_mixing_ratio',
       'sea_salt_aerosol_05_5_mixing_ratio',
       'sea_salt_aerosol_5_20_mixing_ratio',
       'dust_aerosol_003_055_mixing_ratio', 'dust_aerosol_055_09_mixing_ratio',
       'dust_aerosol_09_20_mixing_ratio',
       'hydrophilic_organic_matter_aerosol_mixing_ratio',
       'hydrophobic_organic_matter_aerosol_mixing_ratio',
       'hydrophilic_black_carbon_aerosol_mixing_ratio',
       'hydrophobic_black_carbon_aerosol_mixing_ratio',
       'sulphate_aerosol_mixing_ratio', 'ethane', 'c3h8', 'isoprene',
       'carbon_monoxide_mass_mixing_ratio', 'ozone_mass_mixing_ratio', 'h2o2',
       'formaldehyde', 'hno3', 'nitrogen_monoxide_mass_mixing_ratio',
       'nitrogen_dioxide_mass_mixing_ratio', 'oh', 'organic_nitrates',
       'specific_humidity', 'sul

## Result

In [27]:
summary(final_df).style.background_gradient()

,data type,#missing,Duplicate,#unique,min,max,avg,std dev
total_parking_minutes,float64,0,32,56724,1.050000,44640.000000,30444.295038,8371.362544
metar_temperature_c,float64,0,32,59332,-16.453889,39.650294,16.513721,8.272454
metar_relative_humidity,float64,0,32,59283,1.381661,100.000000,67.245746,18.257788
metar_dew_point_c,float64,0,32,59280,-20.239208,27.066209,11.180438,7.296641
metar_wind_speed_kn,float64,0,32,59265,0.000000,23.125000,7.140583,1.786192
metar_visibility_mi,float64,0,32,59169,0.190000,35.187595,6.352590,1.600460
metar_hour_precipitation,float64,0,32,9743,0.000000,0.104874,0.000813,0.003563
sea_salt_aerosol_003_05_mixing_ratio,float64,0,32,59387,0.000000,0.000000,0.000000,0.000000
sea_salt_aerosol_05_5_mixing_ratio,float64,0,32,59396,0.000000,0.000000,0.000000,0.000000
sea_salt_aerosol_5_20_mixing_ratio,float64,0,32,59410,0.000000,0.000000,0.000000,0.000000


# Pre-processing
## Methods

In [28]:
def get_variable_types(dataframe, exclude=None):
    if exclude is None:
        exclude = []
    continuous_vars = []
    categorical_vars = []

    for column in dataframe.columns:
	    if column not in exclude:
	        categorical_vars.append(column) if dataframe[column].dtype == 'str' else continuous_vars.append(column)

    return continuous_vars, categorical_vars

In [29]:
def confusion():
	plt.figure(figsize=(15, 6))
	conf_matrix = confusion_matrix(y_test, y_pred)
	conf_labels = [f'{i}' for i in range(conf_matrix.shape[0])]
	conf_matrix_df = pd.DataFrame(conf_matrix, columns=conf_labels, index=conf_labels)
	plt.imshow(conf_matrix, interpolation='nearest', cmap=plt.cm.Blues)
	plt.xticks(np.arange(conf_matrix.shape[0]), conf_labels, rotation=45)
	plt.yticks(np.arange(conf_matrix.shape[0]), conf_labels)
	plt.xlabel('Predicted Label')
	plt.ylabel('True Label')
	for i in range(conf_matrix.shape[0]):
	    for j in range(conf_matrix.shape[1]):
	        plt.text(j, i, str(conf_matrix[i, j]), ha='center', va='center', color='black')
	plt.grid(False)
	plt.show()

In [30]:
def feature_importances(model):
	feature_importance = model.feature_importances_
	feature_importance_df = pd.DataFrame({'Feature': X.columns, 'Importance': feature_importance})
	feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)
	plt.figure(figsize=(12, 50))
	sns.barplot(x='Importance', y='Feature', data=feature_importance_df)
	plt.title('Feature Importance')
	plt.xlabel('Importance')
	plt.ylabel('')
	sns.despine(left=True, bottom=True)
	plt.show()

## Model 1
### Encoding

In [32]:
final_df = final_df.drop_duplicates()
final_df

,total_parking_minutes,metar_temperature_c,metar_relative_humidity,metar_dew_point_c,metar_wind_speed_kn,metar_visibility_mi,metar_hour_precipitation,sea_salt_aerosol_003_05_mixing_ratio,sea_salt_aerosol_05_5_mixing_ratio,sea_salt_aerosol_5_20_mixing_ratio,...,hno3,nitrogen_monoxide_mass_mixing_ratio,nitrogen_dioxide_mass_mixing_ratio,oh,organic_nitrates,specific_humidity,sulphur_dioxide_mass_mixing_ratio,temperature,year,month
0,41220.150000,30.718293,30.899519,9.558831,5.710531,2.196935,0.0,1.101830e-11,8.646645e-10,2.827007e-11,...,5.711211e-09,2.694732e-08,3.239839e-08,9.621187e-14,1.688966e-09,0.007535,5.227935e-08,302.226554,2017,4
1,28587.733333,29.397745,78.179996,24.765060,6.290009,2.414297,0.0,2.059410e-10,1.627665e-08,3.240993e-09,...,2.222564e-09,6.977001e-09,1.693530e-08,6.740166e-14,8.613307e-10,0.018132,1.506060e-08,301.692884,2021,7
2,29868.166667,15.522649,76.737610,10.499426,3.765631,1.029053,0.0,7.937799e-12,5.792896e-10,1.267318e-10,...,4.869761e-09,2.471157e-08,4.266760e-08,2.792927e-14,7.472720e-09,0.008187,5.721327e-08,287.567593,2024,1
3,18139.383333,30.951416,42.349314,15.166188,7.073745,5.594613,0.0,2.354342e-11,1.815349e-09,3.219661e-10,...,1.112867e-08,8.965584e-10,4.222114e-08,1.126736e-13,2.468868e-08,0.014630,1.843788e-08,301.816769,2019,5
4,26916.333333,27.367780,81.524175,23.630275,7.467336,2.774049,0.0,1.523394e-10,1.210693e-08,2.119252e-09,...,2.116588e-09,4.818643e-09,1.728175e-08,7.366147e-14,1.265471e-09,0.018309,1.331511e-08,299.727528,2025,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63519,43200.000000,12.047059,82.966676,9.064706,6.873529,5.343588,0.0,3.740082e-11,3.136986e-09,1.782523e-09,...,8.115343e-10,2.796081e-09,1.363158e-08,1.128950e-14,1.126368e-09,0.007033,1.470342e-09,285.244050,2018,11
63520,43200.000000,27.310231,79.124785,22.943894,3.006579,5.950987,0.0,8.937481e-11,6.901622e-09,2.776589e-10,...,5.772846e-11,3.342279e-11,7.785108e-10,1.736884e-15,1.074707e-09,0.017245,5.628162e-10,301.650542,2017,9
63521,44640.000000,26.648810,79.650060,22.431548,3.413690,6.141250,0.0,3.586237e-11,2.777345e-09,2.018195e-10,...,3.031872e-11,2.182628e-11,5.608379e-10,1.656447e-15,7.825175e-10,0.017126,5.198026e-10,300.423651,2018,7
63522,80.266667,26.000000,78.670000,22.000000,2.500000,6.210000,0.0,2.450765e-11,1.920250e-09,2.764327e-10,...,1.985568e-10,7.445210e-11,7.755650e-09,7.959141e-16,4.542368e-09,0.017704,2.995566e-09,299.071251,2025,9


In [34]:
continuous_vars, categorical_vars = get_variable_types(final_df, ['toxin_detected'])
display(continuous_vars, categorical_vars)

['total_parking_minutes',
 'metar_temperature_c',
 'metar_relative_humidity',
 'metar_dew_point_c',
 'metar_wind_speed_kn',
 'metar_visibility_mi',
 'metar_hour_precipitation',
 'sea_salt_aerosol_003_05_mixing_ratio',
 'sea_salt_aerosol_05_5_mixing_ratio',
 'sea_salt_aerosol_5_20_mixing_ratio',
 'dust_aerosol_003_055_mixing_ratio',
 'dust_aerosol_055_09_mixing_ratio',
 'dust_aerosol_09_20_mixing_ratio',
 'hydrophilic_organic_matter_aerosol_mixing_ratio',
 'hydrophobic_organic_matter_aerosol_mixing_ratio',
 'hydrophilic_black_carbon_aerosol_mixing_ratio',
 'hydrophobic_black_carbon_aerosol_mixing_ratio',
 'sulphate_aerosol_mixing_ratio',
 'ethane',
 'c3h8',
 'isoprene',
 'carbon_monoxide_mass_mixing_ratio',
 'ozone_mass_mixing_ratio',
 'h2o2',
 'formaldehyde',
 'hno3',
 'nitrogen_monoxide_mass_mixing_ratio',
 'nitrogen_dioxide_mass_mixing_ratio',
 'oh',
 'organic_nitrates',
 'specific_humidity',
 'sulphur_dioxide_mass_mixing_ratio',
 'temperature',
 'year',
 'month']

[]

In [35]:
sample = final_df.copy()
sample.info()

<class 'pandas.DataFrame'>
Index: 63455 entries, 0 to 63523
Data columns (total 35 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   total_parking_minutes                            63455 non-null  float64
 1   metar_temperature_c                              63455 non-null  float64
 2   metar_relative_humidity                          63455 non-null  float64
 3   metar_dew_point_c                                63455 non-null  float64
 4   metar_wind_speed_kn                              63455 non-null  float64
 5   metar_visibility_mi                              63455 non-null  float64
 6   metar_hour_precipitation                         63455 non-null  float64
 7   sea_salt_aerosol_003_05_mixing_ratio             63455 non-null  float64
 8   sea_salt_aerosol_05_5_mixing_ratio               63455 non-null  float64
 9   sea_salt_aerosol_5_20_mixing_ratio          

In [36]:
sample = pd.get_dummies(sample, columns=categorical_vars, drop_first=True)
print(f'The encoded dataset has {sample.shape[0]} rows and {sample.shape[1]} columns')

The encoded dataset has 63455 rows and 35 columns


In [37]:
sample.columns

Index(['total_parking_minutes', 'metar_temperature_c',
       'metar_relative_humidity', 'metar_dew_point_c', 'metar_wind_speed_kn',
       'metar_visibility_mi', 'metar_hour_precipitation',
       'sea_salt_aerosol_003_05_mixing_ratio',
       'sea_salt_aerosol_05_5_mixing_ratio',
       'sea_salt_aerosol_5_20_mixing_ratio',
       'dust_aerosol_003_055_mixing_ratio', 'dust_aerosol_055_09_mixing_ratio',
       'dust_aerosol_09_20_mixing_ratio',
       'hydrophilic_organic_matter_aerosol_mixing_ratio',
       'hydrophobic_organic_matter_aerosol_mixing_ratio',
       'hydrophilic_black_carbon_aerosol_mixing_ratio',
       'hydrophobic_black_carbon_aerosol_mixing_ratio',
       'sulphate_aerosol_mixing_ratio', 'ethane', 'c3h8', 'isoprene',
       'carbon_monoxide_mass_mixing_ratio', 'ozone_mass_mixing_ratio', 'h2o2',
       'formaldehyde', 'hno3', 'nitrogen_monoxide_mass_mixing_ratio',
       'nitrogen_dioxide_mass_mixing_ratio', 'oh', 'organic_nitrates',
       'specific_humidity', 'sul

### Split

In [ ]:
X = sample.copy()
y = X.pop('toxin_detected')
y_2 = X.pop('toxin_value_standardized_ug_kg')
display(X.shape, y.shape)

In [ ]:
X.columns

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
display(f"Train shape: {X_train.shape}", f"Test shape: {X_test.shape}")

In [ ]:
# showplot('toxin_detected', y)
y_train.value_counts()

In [ ]:
y_train.value_counts()

In [ ]:
# sc = StandardScaler()
# X_train = sc.fit_transform(X_train)
# X_test = sc.transform(X_test)

### Hyperparameters

In [ ]:
# def objective(trial, X, y):
#     # Define parameters to be optimized for the LGBMClassifier
#     param = {
#         "objective": "binary",
#         "metric": "binary_logloss",
#         "verbosity": -1,
#         "boosting_type": "gbdt",
#         "random_state": 42,
#         "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.2),
#         "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
#         "lambda_l1": trial.suggest_float("lambda_l1", 0.005, 0.015),
#         "lambda_l2": trial.suggest_float("lambda_l2", 0.02, 0.06),
#         "max_depth": trial.suggest_int("max_depth", 5, 20),
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.9),
#         "subsample": trial.suggest_float("subsample", 0.8, 1.0),
#         "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
#     }
#
#     cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#     scores = []
#
#     for train_idx, valid_idx in cv.split(X, y):
#         X_tr, X_val = X.iloc[train_idx], X.iloc[valid_idx]
#         y_tr, y_val = y.iloc[train_idx], y.iloc[valid_idx]
#
#         lgbm_classifier = LGBMClassifier(**param)
#         lgbm_classifier.fit(X_tr, y_tr)
#         y_proba = lgbm_classifier.predict_proba(X_val)[:, 1]
#         scores.append(roc_auc_score(y_val, y_proba))
#
#     return np.mean(scores)
#
# # Train Test split
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
#
# # sampler for Optuna optimization using Tree-structured Parzen Estimator sampler
# sampler = optuna.samplers.TPESampler(seed=42)
#
# # Create a study object, run the optimization process and get best parameters
# study = optuna.create_study(direction="maximize", sampler=sampler)
# study.optimize(lambda trial: objective(trial, X, y), n_trials=100)
# best_params = study.best_params
#
# print('='*50)
# print(f"Best params: {best_params}")
# print(f"Best CV AUC: {study.best_value}")

In [ ]:
best_params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "verbosity": -1,
    "boosting_type": "gbdt",
    "random_state": 42,
    'learning_rate': 0.009743961937494949,
    'n_estimators': 213,
    'lambda_l1': 0.013119235781722067,
    'lambda_l2': 0.03749389007648924,
    'max_depth': 18,
    'colsample_bytree': 0.45560483202639046,
    'subsample': 0.8431540916294574,
    'min_child_samples': 8
}

### Build model

In [ ]:
lgbm_classifier = LGBMClassifier(**best_params)
lgbm_classifier.fit(X_train, y_train)
lgbm_classifier.booster_.save_model("toxin_detection_classifier.txt")

y_pred = lgbm_classifier.predict(X_test)
y_pred_proba = lgbm_classifier.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)
f1 = f1_score(y_test, y_pred)

print(f"Test Accuracy: {acc}")
print(f"Test ROC AUC: {auc}")
print(f"Test F1-score: {f1}")

In [ ]:
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

## Confusion matrix

In [ ]:
confusion()

In [ ]:
feature_importances(lgbm_classifier)

## Model 2
### Encoding

In [ ]:
# continuous_vars_2, categorical_vars_2 = get_variable_types(final_df, ['toxin_detected', 'toxin_value_standardized_ug_kg'])
# display(continuous_vars_2, categorical_vars_2)

In [ ]:
# final_df['toxin_value_standardized_ug_kg'] = final_df['toxin_value_standardized_ug_kg'].fillna(0)
# sample = final_df.copy()
# sample['toxin_detected'] == True
# sample.info()

In [ ]:
# sample = pd.get_dummies(sample, columns=categorical_vars_2, drop_first=True)
# print(f'The encoded dataset has {sample.shape[0]} rows and {sample.shape[1]} columns')

### Split

In [ ]:
# X = sample.copy()
# X.columns
# y = X.pop('toxin_value_standardized_ug_kg')
# X.shape
X_2 = X[y]
y_2 = y_2[y]
display(X_2.shape, y_2.shape)

In [ ]:
X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(X_2, y_2, test_size=0.2, random_state=42)
display(f"Train shape: {X_train_2.shape}", f"Test shape: {X_test_2.shape}")

In [ ]:
y_train_2.value_counts()

### Hyperparameters

In [ ]:
# def objective(trial, X, y):
#     # Define parameters to be optimized for the LGBMClassifier
#     param = {
#         "objective": "regression",
#         "metric": "rmse",
#         "verbosity": -1,
#         "boosting_type": "gbdt",
#         "random_state": 42,
#         "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.2),
#         "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
#         "lambda_l1": trial.suggest_float("lambda_l1", 0.005, 0.015),
#         "lambda_l2": trial.suggest_float("lambda_l2", 0.02, 0.06),
#         "max_depth": trial.suggest_int("max_depth", 5, 20),
#         "num_leaves": trial.suggest_int("num_leaves", 7, 255),
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 0.9),
#         "subsample": trial.suggest_float("subsample", 0.8, 1.0),
#         "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
#     }
#
#     cv = KFold(n_splits=5, shuffle=True, random_state=42)
#     scores = []
#
#     for train_idx, valid_idx in cv.split(X_train_2):
#         X_tr, X_val = X_train_2.iloc[train_idx], X_train_2.iloc[valid_idx]
#         y_tr, y_val = y_train_2.iloc[train_idx], y_train_2.iloc[valid_idx]
#
#         lgbm_regressor = LGBMRegressor(**param)
#         lgbm_regressor.fit(X_tr, y_tr)
#         pred = lgbm_regressor.predict(X_val)
#
#         scores.append(mean_squared_error(y_val, pred))
#
#     return np.mean(scores)
#
# # Train Test split
# # X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
#
# # sampler for Optuna optimization using Tree-structured Parzen Estimator sampler
# sampler = optuna.samplers.TPESampler(seed=42)
#
# # Create a study object, run the optimization process and get best parameters
# study = optuna.create_study(direction="maximize", sampler=sampler)
# study.optimize(lambda trial: objective(trial, X_2, y_2), n_trials=50)
# best_params = study.best_params
#
# print('='*50)
# print(f"Best params: {best_params}")

In [ ]:
best_params = {
    "objective": "regression",
    "metric": "rmse",
    "verbosity": -1,
    "boosting_type": "gbdt",
    "random_state": 42,
    'learning_rate': 0.0011634354652036005,
    'n_estimators': 717,
    'lambda_l1': 0.013217597158001457,
    'lambda_l2': 0.03417033228523668,
    'max_depth': 15,
    'num_leaves': 54,
    'colsample_bytree': 0.4581407459012575,
    'subsample': 0.84128095567905,
    'min_child_samples': 48
}

### Build model

In [ ]:
lgbm_regressor = LGBMRegressor(**best_params)
lgbm_regressor.fit(X_train_2, y_train_2)
lgbm_regressor.booster_.save_model("toxin_detection_regressor.txt")

y_pred_2 = lgbm_regressor.predict(X_test_2)

rmse = mean_squared_error(y_test_2, y_pred_2)
mae = mean_absolute_error(y_test_2, y_pred_2)
r2 = r2_score(y_test, y_pred)

print(f"Test RMSE: {rmse}")
print(f"Test MAE: {mae}")
print(f"Test R2-score: {r2}")

In [ ]:
for i in range(len(y_test)):
    print(f"Predicted: {y_pred_2[i]} ; True: {np.array(y_test_2)[i]}")

In [ ]:
min(np.array(y_pred_2))

In [ ]:
confusion()

In [ ]:
feature_importances(lgbm_regressor)